# 15 PyTorch 中 CNN 模型结构怎么表示

上一节从整体上看了 CNN 图像分类案例由哪些部分组成。

这一节只看其中一个部分：

```text
CNN 模型结构怎么表示。
```

注意，这一节不是 API 背诵课。

我们只解决一个入门问题：

```text
在 PyTorch 里，一个 CNN 模型通常由哪些层按什么顺序组成？
每一层大概会怎样改变数据形状？
```

## 1. 为什么要学模型结构表示

前面学习 CNN 时，我们常用文字描述：

```text
卷积层
-> 激活函数
-> 池化层
-> Flatten
-> 全连接层
```

但是到了 PyTorch 里，这些结构需要被写成模型的一部分。

所以现在要把“概念里的层”对应到“模型结构里的层”。

先建立对应关系，后面看代码会轻松很多。

## 2. PyTorch 中的模型可以先理解成一条流水线

一个 CNN 模型可以先理解成一条流水线。

图片从入口进去，一层一层往后走：

```text
输入图片
-> 第一层处理
-> 第二层处理
-> 第三层处理
-> ...
-> 输出类别分数
```

每一层做一件相对明确的事。

前面的层主要提取特征，后面的层主要完成分类。

## 3. Conv2d 对应什么

`Conv2d` 对应前面学过的二维卷积层。

它负责用卷积核在图片或特征图上滑动，提取局部特征。

可以这样理解：

```text
Conv2d：用多个卷积核，从图像中提取多张特征图。
```

它最重要的几个概念不是 API 名字，而是：

```text
输入通道数
输出通道数
卷积核大小
stride
padding
```

这些刚好对应前面已经学过的卷积运算规则。

## 4. 输入通道数和输出通道数怎么理解

在卷积层里，输入通道数表示这一层接收多少张特征图。

比如第一层处理 MNIST 灰度图：

```text
输入通道数 = 1
```

因为 MNIST 图片可以看成 1 x 28 x 28。

输出通道数表示这一层要用多少个卷积核。

比如输出通道数是 6，可以理解成：

```text
这一层用 6 个卷积核
输出 6 张特征图
```

所以要记住：

```text
卷积层输出通道数 = 卷积核个数 = 输出特征图张数
```

## 5. ReLU 对应什么

`ReLU` 对应激活函数。

它通常跟在卷积层或全连接层后面。

可以先理解成：

```text
ReLU：给模型增加非线性表达能力。
```

如果只有卷积层和全连接层，没有激活函数，模型表达能力会弱很多。

ReLU 一般不改变张量形状。

比如：

```text
卷积输出：B x 6 x 24 x 24
经过 ReLU：B x 6 x 24 x 24
```

形状不变，只是里面的数值被处理了。

## 6. MaxPool2d 对应什么

`MaxPool2d` 对应最大池化层。

它负责压缩特征图的高和宽。

可以先理解成：

```text
MaxPool2d：保留局部区域中最明显的特征，同时减小特征图尺寸。
```

比如常见的 2 x 2 最大池化，stride 为 2。

如果输入是：

```text
B x 6 x 24 x 24
```

池化后可能变成：

```text
B x 6 x 12 x 12
```

注意：池化通常不改变通道数，只改变高度和宽度。

## 7. Flatten 对应什么

`Flatten` 对应前面学过的展平操作。

卷积和池化得到的是特征图，形状通常像这样：

```text
B x C x H x W
```

全连接层更习惯接收一维特征向量。

所以需要 Flatten：

```text
B x C x H x W
-> B x (C x H x W)
```

比如：

```text
B x 16 x 4 x 4
-> B x 256
```

Flatten 本身通常没有可学习参数，只是改变数据形状。

## 8. Linear 对应什么

`Linear` 对应全连接层。

在 CNN 分类模型里，全连接层一般放在 Flatten 后面。

它负责根据前面提取出的图像特征，输出类别分数。

比如 10 分类任务，最后一层通常输出 10 个数：

```text
B x 10
```

含义是：

```text
每张图片都有 10 个类别分数。
```

如果 batch 里有 64 张图片，输出形状可以理解成：

```text
64 x 10
```

## 9. 一个简单 CNN 的结构顺序

一个入门 CNN 可以先记成：

```text
Conv2d
-> ReLU
-> MaxPool2d
-> Conv2d
-> ReLU
-> MaxPool2d
-> Flatten
-> Linear
-> ReLU
-> Linear
```

前半部分：

```text
Conv2d + ReLU + MaxPool2d
```

负责提取和压缩图像特征。

后半部分：

```text
Flatten + Linear
```

负责根据特征做分类。

## 10. 为什么经常是卷积、激活、池化放在一起

因为这三个层常常形成一个小组合：

```text
卷积：提取局部特征
激活：增强表达能力
池化：压缩特征图尺寸
```

这组结构可以重复多次。

第一次可能提取简单特征，比如边缘、角点。

后面几次可能提取更复杂的组合特征。

所以 CNN 不是只靠一个卷积层完成分类，而是逐层提取越来越有用的特征。

## 11. 用形状理解模型结构

学 CNN 结构时，不要只盯着层名字。

更重要的是跟踪形状变化。

比如一张 MNIST 图片进入模型，可以大概想成：

```text
B x 1 x 28 x 28
-> 卷积层
B x 6 x 24 x 24
-> 池化层
B x 6 x 12 x 12
-> 卷积层
B x 16 x 8 x 8
-> 池化层
B x 16 x 4 x 4
-> Flatten
B x 256
-> 全连接层
B x 10
```

这里的数字只是一个示例，真实尺寸取决于卷积核大小、stride、padding 和池化设置。

## 12. 为什么 Flatten 后的数字容易算错

CNN 初学者常见错误之一，就是 Flatten 后接 Linear 时维度算错。

因为 Linear 需要知道输入的一维特征长度。

这个长度来自：

$$
C \times H \times W
$$

比如池化后的特征图是：

```text
B x 16 x 4 x 4
```

那么 Flatten 后每张图片的特征长度是：

$$
16 \times 4 \times 4 = 256
$$

所以后面的全连接层输入特征数应该对应 256。

## 13. forward 在模型里表示什么

在 PyTorch 模型中，`forward` 表示前向传播规则。

可以先理解成：

```text
forward 决定图片进入模型后，按什么顺序经过各层。
```

也就是说，定义了层还不够。

还要说明数据怎么从这些层流过去。

从学习角度看，`forward` 就是我们前面一直画的这条线：

```text
输入图片
-> 卷积
-> 激活
-> 池化
-> 展平
-> 全连接
-> 输出类别分数
```

## 14. 这一节暂时不需要背完整代码

这一节的重点不是马上写出完整 CNN 类。

更重要的是先能看懂结构：

```text
Conv2d 是卷积层。
ReLU 是激活函数。
MaxPool2d 是池化层。
Flatten 是展平。
Linear 是全连接层。
forward 是前向传播路线。
```

只要这些关系清楚，后面写 CNN-MNIST 代码时，每一行就都有位置了。

## 15. 本节小结

PyTorch 中的 CNN 模型结构，可以先理解成：

```text
特征提取器 + 分类头
```

特征提取器通常包括：

```text
Conv2d
ReLU
MaxPool2d
```

分类头通常包括：

```text
Flatten
Linear
```

学习模型结构时，最重要的不是背 API，而是能说清楚：

```text
每一层做什么？
输入输出形状怎么变？
哪些层有可学习参数？
```

## 16. 自测问题

1. `Conv2d` 在 CNN 中负责什么？
2. 卷积层的输出通道数和什么有关？
3. `ReLU` 通常会不会改变张量形状？
4. `MaxPool2d` 通常改变通道数吗？它主要改变什么？
5. `Flatten` 为什么要放在全连接层前面？
6. `Linear` 在 CNN 分类模型中通常负责什么？
7. 为什么 Flatten 后接 Linear 时容易出现维度错误？
8. `forward` 可以理解成什么？